In [ ]:
import os

os.environ["ELSEVIER_API_KEY"] = ""  #Enter your api key here
#os.environ['ELSEVIER_INST_TOKEN']=''
API_KEY = os.getenv("ELSEVIER_API_KEY")
INST_TOKEN = os.getenv("ELSEVIER_INST_TOKEN")  # We don't need it now

In [6]:
import os
import time
import json
import csv
import requests
from typing import Dict, List, Any, Optional

if not API_KEY:
    raise RuntimeError("Missing ELSEVIER_API_KEY env var.")

BASE = "https://api.elsevier.com"
SEARCH_URL = f"{BASE}/content/search/scopus"

HEADERS = {
    "X-ELS-APIKey": API_KEY,
    "Accept": "application/json",
}
if INST_TOKEN:
    HEADERS["X-ELS-Insttoken"] = INST_TOKEN

def safe_get(url: str, params: Dict[str, Any], max_retry: int = 6) -> Dict[str, Any]:
    backoff = 1.5
    for attempt in range(max_retry):
        r = requests.get(url, headers=HEADERS, params=params, timeout=60)
        if r.status_code == 200:
            return r.json()

        if r.status_code in (429, 500, 502, 503, 504):
            wait = backoff ** attempt
            print(f"[WARN] {r.status_code} retry in {wait:.1f}s ...")
            time.sleep(wait)
            continue

        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:500]}")
    raise RuntimeError("Exceeded retries")

def extract_entry(e: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "eid": e.get("eid"),
        "doi": e.get("prism:doi"),
        "title": e.get("dc:title"),
        "coverDate": e.get("prism:coverDate"),
        "year": (e.get("prism:coverDate") or "")[:4] if e.get("prism:coverDate") else None,
        "publicationName": e.get("prism:publicationName"),
        "issn": e.get("prism:issn"),
        "volume": e.get("prism:volume"),
        "issueIdentifier": e.get("prism:issueIdentifier"),
        "pageRange": e.get("prism:pageRange"),
        "citedby_count": e.get("citedby-count"),
        "type": e.get("subtypeDescription") or e.get("prism:aggregationType"),
        "authors": e.get("dc:creator"),  
        "link_scopus": next((l.get("@href") for l in e.get("link", []) if l.get("@ref") == "scopus"), None),
    }

def scopus_search_all(query: str, count: int = 25, max_records: Optional[int] = None) -> List[Dict[str, Any]]:
    start = 0
    seen_eid = set()
    results: List[Dict[str, Any]] = []

    while True:
        params = {
            "query": query,
            "count": count,
            "start": start,
            "view": "STANDARD",  
        }
        data = safe_get(SEARCH_URL, params=params)

        sr = data.get("search-results", {})
        entries = sr.get("entry", []) or []
        if not entries:
            break

        for e in entries:
            rec = extract_entry(e)
            eid = rec.get("eid")
            if eid and eid not in seen_eid:
                seen_eid.add(eid)
                results.append(rec)

        total = int(sr.get("opensearch:totalResults", "0") or 0)
        start += count

        print(f"[INFO] fetched {len(results)} / total {total} (start={start})")

        if max_records and len(results) >= max_records:
            results = results[:max_records]
            break
        if start >= total:
            break

        time.sleep(0.2)  #Avoid being banned

    return results

def save_jsonl(path: str, rows: List[Dict[str, Any]]):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def save_csv(path: str, rows: List[Dict[str, Any]]):
    if not rows:
        return
    cols = list(rows[0].keys())
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        w.writerows(rows)

if __name__ == "__main__":
    QUERY = (
        'TITLE-ABS-KEY( polysaccharide OR glycan OR glycans OR "glycosaminoglycan" OR GAG OR chitosan OR cellulose OR alginate OR hyaluronan ) '
        'AND TITLE-ABS-KEY( modulus OR tensile OR "Young*" OR rheology OR viscosity OR "storage modulus" OR "loss modulus" OR Tg OR DSC OR TGA OR swelling )'
    )  #Change to your own key words

    out_dir = "scopus_dump" #Can rename the folder
    os.makedirs(out_dir, exist_ok=True)

    rows = scopus_search_all(QUERY, count=25, max_records=200)  # max_records means the number of paper you want to get
    save_jsonl(os.path.join(out_dir, "scopus_search.jsonl"), rows)
    save_csv(os.path.join(out_dir, "scopus_search.csv"), rows)

    print(f"[DONE] saved {len(rows)} records to {out_dir}/")

[INFO] fetched 25 / total 121718 (start=25)
[INFO] fetched 50 / total 121718 (start=50)
[INFO] fetched 75 / total 121718 (start=75)
[INFO] fetched 100 / total 121718 (start=100)
[INFO] fetched 125 / total 121718 (start=125)
[INFO] fetched 150 / total 121718 (start=150)
[INFO] fetched 175 / total 121718 (start=175)
[INFO] fetched 200 / total 121718 (start=200)
[DONE] saved 200 records to scopus_dump/


In [8]:
import pandas as pd

df = pd.read_csv("scopus_dump/scopus_search.csv")

column = df["doi"]

#print(column)

doi = df["doi"].tolist()

doi_10=doi[11:100]

#print(doi_10)

In [ ]:
import os
import requests
import time

os.makedirs("papers_xml", exist_ok=True)

headers = {
    "X-ELS-APIKey": API_KEY,
    "Accept": "application/xml"
}

for doi in doi_10:
    url = f"https://api.elsevier.com/content/article/doi/{doi}"
    
    r = requests.get(url, headers=headers)
    print(doi, r.status_code)

    if r.status_code == 200:  #if the request is successful, save the XML content to a file
        filename = doi.replace("/", "_") + ".xml"
        with open(f"papers_xml/{filename}", "wb") as f:
            f.write(r.content)

    time.sleep(0.2)  # Avoid being banned

10.1038/s41598-026-38254-8 404
10.1007/s43939-025-00530-1 404
10.1038/s41598-025-29706-8 404
10.1038/s41598-025-31548-3 404
10.1007/s44187-025-00800-0 404
10.1038/s41598-025-32760-x 404
10.1038/s41598-026-38234-y 404
10.1007/s10856-025-06988-y 404
10.1038/s41598-025-31372-9 404
10.1038/s41467-026-68803-8 404
10.1007/s10103-026-04806-7 404
10.1186/s13023-025-04157-6 404
10.1038/s41467-025-68038-z 404
10.1186/s12951-025-03816-x 404
10.1038/s41467-025-67035-6 404
10.1007/s44187-025-00775-y 404
10.1186/s43088-025-00725-8 404
nan 404
10.1007/s00418-026-02460-2 404
10.1038/s41598-026-35333-8 404
10.1007/s40820-025-02039-x 404
10.1038/s41598-025-32332-z 404
10.1038/s41598-026-35665-5 404
10.1007/s00404-026-08348-9 404
10.1186/s13065-025-01697-7 404
10.1186/s12985-026-03080-x 404
10.1007/s12155-025-10931-y 404
10.1007/s40820-025-02046-y 404
10.1038/s41467-026-69037-4 404
10.1038/s41377-025-02083-7 404
10.1186/s13020-026-01325-z 404
10.1186/s12951-025-03969-9 404
10.1186/s42825-025-00224-7 404
